# `sgp_set` for chest X-ray pathology detection

**Task.** Binary detection of a single pathology on NIH ChestX-ray14, with a
DenseNet-121 from `torchxrayvision` **trained on CheXpert** — so ChestX-ray14 is
data the classifier never saw. No training here, inference only.

**Output.** A pickled `DataFrame` with `y_true`, `y_pred`, `kappa` (softmax
response, in `(0.5, 1)`), directly usable with `theta_min, theta_max = 0.5, 1`.

**Requirements.** `pip install torchxrayvision`, and the updated `nb_setup.py`.
NIH images (`images_001.tar.gz` ... `images_012.tar.gz`) from
<https://nihcc.app.box.com/v/ChestXray-NIHCC>, extracted into one flat folder.
The label CSV ships inside `torchxrayvision`, nothing else to download.

**Two choices that matter.** (i) One film per patient: ChestX-ray14 has 112,120
images for 30,805 patients, so the raw set is not i.i.d. Pneumothorax patients are
imaged repeatedly (chest tube, re-expansion follow-up), so the 4.7% image-level
rate falls to 1.5% once each patient contributes a single random film. The sampler
below therefore takes a *positive* film where the patient has one, which keeps one
row per patient while recovering every affected patient. (ii) The operating
threshold is fitted on a slice held out from `Sn`, then folded into the head's
bias — `kappa` is the softmax response of the *shifted* head, so it is
consistent with `y_pred` by construction.

In [ ]:
%run -i ../../python_scripts/nb_setup.py

In [ ]:
IMGPATH = "data/ChestXray-NIHCC/images"  # flat folder of the NIH .png files
WEIGHTS = "densenet121-res224-chex"      # CheXpert-trained -> NIH is unseen
PATHOLOGY = "Pneumothorax"               # revisit after the patient-level table below
N_MAX = 30_000                           # cap on images pushed through the net
P_TAU = 0.2                              # share of samples reserved to fit the threshold
SEED, OUT = 0, "sgp_set_xrv_SR"

rng = np.random.default_rng(SEED)
torch.set_num_threads(os.cpu_count())

In [ ]:
# Dataset, with labels re-ordered to match the model's output columns.
# unique_patients=False: xrv would otherwise keep each patient's *first* film,
# which under-represents findings that appear on follow-ups (pneumothorax).
model = xrv.models.DenseNet(weights=WEIGHTS).eval()
tf = torchvision.transforms.Compose(
    [xrv.datasets.XRayCenterCrop(), xrv.datasets.XRayResizer(224)]
)
d = xrv.datasets.NIH_Dataset(
    imgpath=IMGPATH, transform=tf, views=["PA", "AP"], unique_patients=False
)
xrv.datasets.relabel_dataset(model.pathologies, d)

# Labels come from the CSV, so this cell is instant. "patients" is the number of
# positives the sampler below will return for each pathology: pick PATHOLOGY from
# it before paying for the forward pass.
csv = d.csv.reset_index(drop=True)
n_pat = csv["Patient ID"].nunique()
lab = pd.DataFrame(
    d.labels, columns=[p or f"_{i}" for i, p in enumerate(model.pathologies)]
)
tab = pd.DataFrame(
    {
        "images": lab.sum(),
        "patients": lab.groupby(csv["Patient ID"].values).max().sum(),
    }
).astype(int)
tab = tab[tab.patients > 0].sort_values("patients", ascending=False)
print(len(d), "images |", n_pat, "patients")
print(tab.assign(patient_rate=(tab.patients / n_pat).round(4)))

In [ ]:
assert PATHOLOGY in model.pathologies, model.pathologies
col = model.pathologies.index(PATHOLOGY)

# One film per patient: a positive one where the patient has any, else at random.
# Membership still depends on that patient's own data only, so the retained pairs
# stay independent and Prop. 1 applies -- to an enriched P (see the closing note).
pos = pd.Series(d.labels[:, col], index=csv.index).fillna(0)
order = (
    csv.assign(_pos=pos)
    .sample(frac=1, random_state=SEED)
    .sort_values("_pos", ascending=False, kind="stable")
)
idx = order.drop_duplicates("Patient ID").index.to_numpy()
sel = rng.permutation(idx)[:N_MAX]
ds = xrv.datasets.SubsetDataset(d, sel)
n_pos = int(np.nansum(d.labels[sel, col]))
print(f"{PATHOLOGY}: N={len(ds)} | positives={n_pos} ({n_pos/len(ds):.4f})")
assert n_pos > 500, "positive stratum too thin for a useful FNR/SE bound"

In [ ]:
# Forward pass (~15-40 min on a laptop CPU for 30k images; PNG decoding dominates)
loader = torch.utils.data.DataLoader(ds, batch_size=32, num_workers=4)
p, y = [], []
with torch.no_grad():
    for b in tqdm(loader):
        p.append(model(b["img"]).numpy()[:, col])
        y.append(b["lab"].numpy()[:, col])
p, y = np.concatenate(p), np.concatenate(y)

m = ~np.isnan(y)
p, y = p[m], y[m].astype(int)
print("N =", len(y), "| AUC =", round(roc_auc_score(y, p), 3))

In [ ]:
# Threshold fitted on a disjoint slice (Youden's J), then absorbed into the head's bias
tau_split = rng.random(len(y)) < P_TAU
fpr, tpr, thr = roc_curve(y[tau_split], p[tau_split])
tau = float(np.clip(thr[np.argmax(tpr - fpr)], 1e-6, 1 - 1e-6))

s = logit(np.clip(p, 1e-6, 1 - 1e-6)) - logit(tau)  # decision margin of the shifted head
p_shift = expit(s)
sgp_df = pd.DataFrame(
    {
        "y_true": y.astype(float),
        "y_pred": (s >= 0).astype(float),
        "kappa": np.maximum(p_shift, 1 - p_shift),  # softmax response
    }
)[~tau_split].reset_index(drop=True)
print("tau =", round(tau, 4))

In [ ]:
def report(df, name):
    n = len(df)
    err = (df.y_pred != df.y_true).mean()
    fp = ((df.y_pred == 1) & (df.y_true == 0)).sum()
    fn = ((df.y_pred == 0) & (df.y_true == 1)).sum()
    print(
        f"{name}: N={n} | positives={df.y_true.mean():.3f} | 0/1 risk={err:.3f} | "
        f"FP={fp} FN={fn} | FPR={fp/(df.y_true==0).sum():.3f} "
        f"FNR={fn/(df.y_true==1).sum():.3f} | "
        f"kappa in [{df.kappa.min():.3f}, {df.kappa.max():.3f}]"
    )


report(sgp_df, "sgp_set")
pickle.dump(sgp_df, open(OUT, "wb"))
sgp_df.head(3)

Plug into `individual_control.ipynb` / `joint_control.ipynb`:

```python
sgp_df_SR = pickle.load(open("sgp_set_xrv_SR", "rb"))
theta_min_SR, theta_max_SR = 0.5, 1
```

Caveats to keep in mind when reporting. The cohort is **enriched**: sampling a
positive film where one exists raises prevalence well above the 1.5% of a
one-random-film-per-patient cohort. FPR, FNR, SE and SP are per-class rates and so
are unaffected, but the FP/FN risks and PPV are relative to that inflated
prevalence and should be labelled accordingly -- the same deliberate
class-balance manipulation as your Section 6.3. Also, NIH labels are NLP-mined
from radiology reports, so `y_true` carries label noise, and the CheXpert -> NIH
transfer means the operating point is set under distribution shift. Neither breaks the bounds
(they are distribution-free and `f` is arbitrary), but both affect the *baseline*
numbers you quote at full coverage.